# Daily temperature by price area Gold

Create daily average temperatures for each Norwegian electricity price area using historical Frost weather observations.

Use temperature measurements taken 2 metres above ground with quality code 0.

Prefer hourly (PT1H) observations. If unavailable, calculate an hourly average from at least five valid 10 minute (PT10M) readings.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from gold_layer.temperature import ingest

In [2]:
spark = (
    SparkSession.builder.appName("Gold Temperature")
    .master("local[4]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

26/11/21 21:22:47 WARN Utils: Your hostname, homelab resolves to a loopback address: 127.0.1.1; using 192.168.10.53 instead (on interface lan0)
26/11/21 21:22:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/21 21:22:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
result = ingest(
    silver_root=PROJECT_ROOT / "data/silver",
    gold_root=PROJECT_ROOT / "data/gold",
    start="2024-01-01",
    end="2026-01-01",
    spark=spark,
)

print(result)

{'dataset': 'daily_temperature', 'status': 'success', 'rows': 3655, 'gold_path': '/home/roland/UiS/rewrite/group14/data/gold/temperature/daily'}


In [4]:
daily = spark.read.parquet(result["gold_path"])
daily.printSchema()
daily.orderBy("date", "price_area").show(15, truncate=False)

root
 |-- date: date (nullable = true)
 |-- price_area: string (nullable = true)
 |-- avg_temperature_c: decimal(34,22) (nullable = true)
 |-- station_count: long (nullable = true)
 |-- station_hours_pt1h: long (nullable = true)
 |-- station_hours_pt10m_fallback: long (nullable = true)
 |-- avg_station_hours: double (nullable = true)

+----------+----------+--------------------------+-------------+------------------+----------------------------+------------------+
|date      |price_area|avg_temperature_c         |station_count|station_hours_pt1h|station_hours_pt10m_fallback|avg_station_hours |
+----------+----------+--------------------------+-------------+------------------+----------------------------+------------------+
|2024-01-01|NO1       |-6.5029282308832107473537 |164          |1701              |2039                        |22.804878048780488|
|2024-01-01|NO2       |-2.7761252864091909101627 |166          |2196              |1605                        |22.897590361445783|
|20

## Coverage

In [5]:
print("Gold rows:", daily.count())
print("Distinct area/dates:", daily.select("date", "price_area").distinct().count())
print(
    "Null key/temperature rows:",
    daily.filter(
        F.col("date").isNull()
        | F.col("price_area").isNull()
        | F.col("avg_temperature_c").isNull()
    ).count(),
)

daily.groupBy("price_area").agg(
    F.count("*").alias("days"),
    F.min("station_count").alias("min_stations"),
    F.round(F.avg("station_count"), 1).alias("avg_stations"),
    F.max("station_count").alias("max_stations"),
    F.sum("station_hours_pt1h").alias("hours_from_pt1h"),
    F.sum("station_hours_pt10m_fallback").alias("hours_from_fallback"),
).orderBy("price_area").show(truncate=False)

Gold rows: 3655
Distinct area/dates: 3655
Null key/temperature rows: 0
+----------+----+------------+------------+------------+---------------+-------------------+
|price_area|days|min_stations|avg_stations|max_stations|hours_from_pt1h|hours_from_fallback|
+----------+----+------------+------------+------------+---------------+-------------------+
|NO1       |731 |75          |164.0       |172         |1338230        |1514988            |
|NO2       |731 |105         |175.7       |190         |1875482        |1172318            |
|NO3       |731 |98          |189.0       |209         |1701336        |1587510            |
|NO4       |731 |129         |194.4       |237         |2216704        |1146434            |
|NO5       |731 |45          |92.4        |102         |789548         |814758             |
+----------+----+------------+------------+------------+---------------+-------------------+

